# Lesson 3: Agentic Search

In [1]:
!pip install -q tavily-python duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.2 MB/s eta 0:00:00


In [2]:
import os
from tavily import TavilyClient
os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [3]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]

"Nvidia's new Blackwell B200 GPU is the company's flagship AI product, featuring 208 billion transistors and designed to reduce AI inference operating costs and energy consumption by up to 25 times compared to the H100. The Blackwell architecture is set to power the RTX 50-series graphics cards and includes a Decompression Engine for accessing massive amounts of memory in the NVIDIA Grace™ CPU. Major organizations like Amazon Web Services, Google, Meta, Microsoft, and others are expected to adopt the Blackwell platform for AI applications."

## Regular search

In [4]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [5]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"retornando resultados anteriores devido à exceção atingir ddg.")
        results = [ # capa de caso onde os limites de taxa DDG devido ao alto volume deeplearning.ai
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results


for i in search(query):
    print(i)

https://weather.com/weather/today/l/San+Francisco+CA+USCA0987
https://weather.com/weather/tenday/l/San+Francisco+CA+USCA0987:1:US


In [6]:
def scrape_weather_info(url):
    """Extrair conteúdo do URL fornecido"""
    if not url:
        return "Não foi possível encontrar informações meteorológicas."

    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Falha ao recuperar a página da web."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [ ]:
# use o DuckDuckGo para encontrar sites e pegue o primeiro resultado
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

In [8]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/San+Francisco+CA+USCA0987


recents Weather Forecasts Radar & Maps News & Media Products Health & Wellness Account Lifestyle Privacy Specialty Forecasts San Francisco, CA Coastal Flood Advisory +1 More Today's Forecast for San Francisco, CA Morning Afternoon Evening Overnight Weather Today in San Francisco, CA 6:51 am 4:57 pm Don't Miss Hourly Forecast Now 6 am 7 am 8 am 9 am Outside That's Not What Was Expected Daily Forecast Today Sun 17 Mon 18 Tue 19 Wed 20 Radar We Love Our Critters Seasonal Skin Solutions Home, Garage & Garden Cold Weather Fabrics Health News For You Happening Near San Francisco, CA Popular Nextdoor posts Rain Or Shine, It's Playtime Stay Safe Expert Advice Weather in your inbox Your local forecast, plus daily trivia, stunning photos and our meteorologists’ top picks. All in one place, every weekday morning. By signing up, you're opting in to receive the Morning Brief email newsletter. To manage your data, visit Data Ri

## Agentic Search

In [9]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1731763830, 'localtime': '2024-11-16 05:30'}, 'current': {'last_updated_epoch': 1731762900, 'last_updated': '2024-11-16 05:15', 'temp_c': 9.4, 'temp_f': 48.9, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 5.8, 'wind_kph': 9.4, 'wind_degree': 347, 'wind_dir': 'NNW', 'pressure_mb': 1018.0, 'pressure_in': 30.05, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 66, 'cloud': 0, 'feelslike_c': 8.0, 'feelslike_f': 46.5, 'windchill_c': 8.5, 'windchill_f': 47.3, 'heatindex_c': 9.3, 'heatindex_f': 48.8, 'dewpoint_c': 5.0, 'dewpoint_f': 41.0, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 9.6, 'gust_kph': 15.5}}


In [10]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)

{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1731763830,
        "localtime": "2024-11-16 05:30"
    },
    "current": {
        "last_updated_epoch": 1731762900,
        "last_updated": "2024-11-16 05:15",
        "temp_c": 9.4,
        "temp_f": 48.9,
        "is_day": 0,
        "condition": {
            "text": "Clear",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/113.png",
            "code": 1000
        },
        "wind_mph": 5.8,
        "wind_kph": 9.4,
        "wind_degree": 347,
        "wind_dir": "NNW",
        "pressure_mb": 1018.0,
        "pressure_in": 30.05,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 66,
        "cloud": 0,
        "feelslike_c": 8.0,
        "feelslike_f": 46.5,
        "windchill_c": 8.5,
        "windch

<img src="./google_sample.png" width="800" height="600">